In [1]:
# start date 27/02/2025

import numpy as np
import math
import pandas as pd
import cv2
import os
import re
from pdf2image import convert_from_path 
#import tqdm
#from scipy.io import loadmat

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

from PIL import Image
# import pytesseract

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from keras import backend as K

# from utils import *

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from keras.layers import *

# from keras.applications import MobileNetV2
# from keras.applications import InceptionResNetV2

from keras.models import Model
from keras.models import model_from_json
from keras import regularizers

from ultralytics import YOLO

#from keras.initializers import he_normal

from keras.models import load_model
from math import sqrt


In [2]:
# Extracting BOM from a .jpg single drawing :

# Load the trained YOLOv8 model
model = YOLO("runs/detect/train9/weights/best.pt")

# Path to your test image
#test_image = "/Users/subrata/workstation/jupyterFiles/testdrg/SM-MFD-5KW-003-23.jpg" 
#test_image = "/Users/subrata/workstation/jupyterFiles/testdrg/8992-18-OAC-02-04-Model-page-001.jpg"  
test_image = "/Users/subrata/workstation/jupyterFiles/testdrg_jpg/EMM1-15-RC105-2-Model-page-001.jpg"

# Run inference
results = model(test_image, save=True, conf=0.5)
x1 = 0
x2 = 0
y1 = 0
y2 = 0
# Load the image using OpenCV

image = cv2.imread(test_image)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

# Process detection results
for result in results:
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Bounding box coordinates
        class_id = int(box.cls[0])  # Class index
        confidence = float(box.conf[0])  # Confidence score

        # Crop the detected object
        cropped_obj = image[y1:y2, x1:x2]
        
        # Define box color (green for visibility)
        color = (0, 255, 0)  # RGB (Green)

        # Draw the bounding box
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)
        """
        # Display class name and confidence
        label = f"{model.names[class_id]} {confidence:.2f}"
        cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        """
# Show the result image
if x1 == 0:
    print("There is no bom")
else:
    x_pil = Image.fromarray(image)
    display(x_pil)




image 1/1 /Users/subrata/workstation/jupyterFiles/testdrg_jpg/EMM1-15-RC105-2-Model-page-001.jpg: 480x640 (no detections), 72.8ms
Speed: 4.2ms preprocess, 72.8ms inference, 3.8ms postprocess per image at shape (1, 3, 480, 640)
Results saved to runs/detect/predict59
There is no bom


In [5]:
## Extracting BOM from the test drawings with .jpg files using trained model :

# Load the trained YOLOv8 model
model = YOLO("runs/detect/train9/weights/best.pt")

# Path to test folder (containing test images)

test_folder = "/Users/subrata/workstation/jupyterFiles/testdrg_jpg/"

output_folder = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/pred_cropped_bom_jpg/"  # Folder to save cropped images
os.makedirs(output_folder, exist_ok=True)  # Create folder if not exists

# List of images where 'bom' was not detected
no_bom_drgs = []

# Process each image in the test folder
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):  # Check for image files
        test_image = os.path.join(test_folder, filename)

        # Run inference
        results = model(test_image, conf=0.5)
        x1 = 0
        y1 = 0
        x2 = 0
        y2 = 0
        crop_count = 0

        # Load the image using OpenCV
        image = cv2.imread(test_image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

        # Extract filename (without extension)
        image_name = os.path.splitext(filename)[0]

        # Process detection results
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])  # Bounding box coordinates
                class_id = int(box.cls[0])  # Class index
                confidence = float(box.conf[0])  # Confidence score

                # Crop the detected object
                cropped_obj = image[y1:(y2+10), x1:(x2+10)]

                # Save the cropped image
                cropped_filename_original = f"{output_folder}/{image_name}_crop_{crop_count}.jpg"
                
                cropped_filename = re.sub(r'_crop_\d+', '', cropped_filename_original)

                cv2.imwrite(cropped_filename, cv2.cvtColor(cropped_obj, cv2.COLOR_RGB2BGR))  # Convert RGB to BGR for saving
                crop_count += 1

        # If no 'bom' is detected, add the image to the list
        if x1 == 0:
            no_bom_drgs.append(filename)
            print(f"No 'bom' detected in {filename}.")

# Save the list of 'no bom' images to a text file
no_bom_file = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/pred_no_bom_jpg.txt"
with open(no_bom_file, "w") as f:
    for img in no_bom_drgs:
        f.write(img + "\n")

print(f"\n📂 List of 'no bom' images saved to {no_bom_file}")
print(f"📂 Cropped images saved in {output_folder}")






image 1/1 /Users/subrata/workstation/jupyterFiles/testdrg_jpg/0085-16-SWZ-00-00 R2-Model-page-001.jpg: 480x640 1 bom, 66.1ms
Speed: 8.6ms preprocess, 66.1ms inference, 10.8ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/subrata/workstation/jupyterFiles/testdrg_jpg/1789073R0_MAIN_FRAME_ASSY_1 (SHT.1)-page-001.jpg: 480x640 (no detections), 60.5ms
Speed: 4.3ms preprocess, 60.5ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in 1789073R0_MAIN_FRAME_ASSY_1 (SHT.1)-page-001.jpg.

image 1/1 /Users/subrata/workstation/jupyterFiles/testdrg_jpg/1794039R1 (1 OF 4)_TRIPPER FRAME (FRONT PART)-page-001.jpg: 480x640 (no detections), 55.1ms
Speed: 3.8ms preprocess, 55.1ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in 1794039R1 (1 OF 4)_TRIPPER FRAME (FRONT PART)-page-001.jpg.

image 1/1 /Users/subrata/workstation/jupyterFiles/testdrg_jpg/1789073R0_MAIN_FRAME_ASSY_1 (SHT.2)-page-001.jpg: 480x640 1 bom, 53

In [6]:
## Extracting BOM from the test drawings with .pdf files using trained model :

## Load the trained YOLOv8 model
model = YOLO("runs/detect/train9/weights/best.pt")

# Path to test folder (containing test PDFs)
test_folder = "/Users/subrata/workstation/jupyterFiles/testdrg_pdf/"
output_folder = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/pred_cropped_bom_pdf/"  # Folder to save cropped images
os.makedirs(output_folder, exist_ok=True)  # Create folder if not exists

# List of PDFs where 'bom' was not detected
no_bom_drgs = []

# Process each PDF in the test folder
for filename in os.listdir(test_folder):
    if filename.lower().endswith(".pdf"):  # Check for PDF files
        pdf_path = os.path.join(test_folder, filename)

        # Convert PDF to images
        images = convert_from_path(pdf_path, dpi=300)  # Higher DPI for better OCR and detection

        pdf_has_bom = False  # Flag to track if BOM is detected in any page

        # Process each page
        for page_num, image in enumerate(images):
            # Convert PIL image to OpenCV format
            image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

            # Run YOLO inference
            results = model(image_cv, conf=0.5)

            x1, y1, x2, y2 = 0, 0, 0, 0
            crop_count = 0

            # Process detection results
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])  # Bounding box coordinates
                    class_id = int(box.cls[0])  # Class index
                    confidence = float(box.conf[0])  # Confidence score

                    # Crop the detected object
                    cropped_obj = image_cv[y1:(y2+10), x1:(x2+10)]

                    # Save the cropped image
                    cropped_filename = f"{output_folder}/{os.path.splitext(filename)[0]}_crop_{crop_count}.jpg"
                    cv2.imwrite(cropped_filename, cropped_obj)  # Save as BGR format
                    crop_count += 1
                    pdf_has_bom = True  # Mark that BOM was found

        # If no BOM was detected in any page, add to the list
        if not pdf_has_bom:
            no_bom_drgs.append(filename)
            print(f"No 'bom' detected in {filename}.")

# Save the list of PDFs with no BOM detected
no_bom_file = "/Users/subrata/workstation/jupyterFiles/yolo_data_file/pred_no_bom_pdf.txt"
with open(no_bom_file, "w") as f:
    for pdf in no_bom_drgs:
        f.write(pdf + "\n")

print(f"\n📂 List of 'no bom' PDFs saved to {no_bom_file}")
print(f"📂 Cropped images saved in {output_folder}")



0: 480x640 1 bom, 73.7ms
Speed: 5.6ms preprocess, 73.7ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 66.8ms
Speed: 3.5ms preprocess, 66.8ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in 1789074R0_MAIN_FRAME_ASSY_2 (SHT.1).pdf.

0: 480x640 (no detections), 76.7ms
Speed: 2.5ms preprocess, 76.7ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in 1-JM03-VD00-1-0002_BW-1600_12°_90Lbs).pdf.

0: 480x640 1 bom, 77.8ms
Speed: 3.6ms preprocess, 77.8ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 71.3ms
Speed: 3.9ms preprocess, 71.3ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in 1794039R1 (4 OF 4)_TRIPPER FRAME (FRONT PART).pdf.

0: 480x640 (no detections), 74.4ms
Speed: 3.9ms preprocess, 74.4ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in 179